In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week7-lesson-4"). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [2]:
#order = 'order_id long , order_date date , cust_id long, status string'

In [3]:
orderSchema = StructType([
    StructField("order_id",LongType()),
    StructField("order_date",DateType()),
    StructField("cust_id",LongType()),
    StructField("status",StringType()),
])

In [4]:
df1 = spark.read.schema(orderSchema).option("header","false").csv('/public/trendytech/orders/orders_1gb.csv')

In [5]:
df1.show(4)

+--------+----------+-------+---------------+
|order_id|order_date|cust_id|         status|
+--------+----------+-------+---------------+
|       1|2013-07-25|  11599|         CLOSED|
|       2|2013-07-25|    256|PENDING_PAYMENT|
|       3|2013-07-25|  12111|       COMPLETE|
|       4|2013-07-25|   8827|         CLOSED|
+--------+----------+-------+---------------+
only showing top 4 rows



### Cache spark Tables - Eager

In [6]:
spark.sql("use itv024128")

""


In [7]:
spark.sql("show tables")

database,tableName,isTemporary
itv024128,groceries,false
itv024128,groceries_ext,false
itv024128,groceries_ext_json,false
itv024128,groceries_json,false
itv024128,orders1gb,false
itv024128,orders_ext,false


In [8]:
spark.sql(" drop table itv024128.orders1GB ")

""


In [9]:
df1.write.format('csv').saveAsTable('itv024128.orders1GB')

In [10]:
spark.sql("select count(*) from itv024128.orders1GB ")

count(1)
25831125


In [11]:
spark.sql("cache table itv024128.orders1GB") ## this is not lazy be default unlike df.cache. also caches entire table, irrespective of next query on the table

""


In [12]:
spark.sql("select count(*) from itv024128.orders1GB ")  ## this runs very fast as the table is cacched entirely

count(1)
25831125


In [13]:
spark.sql("select distinct status from itv024128.orders1GB ") 

status
PENDING_PAYMENT
COMPLETE
ON_HOLD
PAYMENT_REVIEW
PROCESSING
CLOSED
SUSPECTED_FRAUD
PENDING
CANCELED


In [14]:
spark.sql("select count (distinct status) from itv024128.orders1GB ") 

count(DISTINCT status)
9


### Remove table cache

In [15]:
spark.sql("uncache table itv024128.orders1GB ") 

""


In [16]:
spark.sql("select count (distinct status) from itv024128.orders1GB ") ## this will run slow compared to cache run

count(DISTINCT status)
9


### Lazy table cache

In [17]:
spark.sql("cache lazy table itv024128.orders1GB")  ## not cached now

""


In [18]:
spark.sql("select count (distinct status)  from itv024128.orders1GB ") ## cache initiated

count(DISTINCT status)
9


In [19]:
spark.sql("select count(*) from itv024128.orders1GB ") ## runs fast

count(1)
25831125


In [20]:
spark.sql("select distinct status from itv024128.orders1GB ").show()

+---------------+
|         status|
+---------------+
|PENDING_PAYMENT|
|       COMPLETE|
|        ON_HOLD|
| PAYMENT_REVIEW|
|     PROCESSING|
|         CLOSED|
|SUSPECTED_FRAUD|
|        PENDING|
|       CANCELED|
+---------------+



In [21]:
spark.sql("select  status,count(*) from itv024128.orders1GB group by status ").show()

+---------------+--------+
|         status|count(1)|
+---------------+--------+
|PENDING_PAYMENT| 5636250|
|       COMPLETE| 8587125|
|        ON_HOLD| 1424250|
| PAYMENT_REVIEW|  273375|
|     PROCESSING| 3103125|
|         CLOSED| 2833500|
|SUSPECTED_FRAUD|  584250|
|        PENDING| 2853750|
|       CANCELED|  535500|
+---------------+--------+



In [22]:
spark.sql("select * from itv024128.orders1GB limit 5").show()

+--------+----------+-------+----------+
|order_id|order_date|cust_id|    status|
+--------+----------+-------+----------+
|   51049|2014-06-09|   4983|PROCESSING|
|   51050|2014-06-09|   1840|   ON_HOLD|
|   51051|2014-06-09|   8207|  COMPLETE|
|   51052|2014-06-09|   6254|  COMPLETE|
|   51053|2014-06-09|    348|   PENDING|
+--------+----------+-------+----------+



In [23]:
spark.sql("describe extended itv024128.orders1GB").show()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|            order_id|              bigint|   null|
|          order_date|                date|   null|
|             cust_id|              bigint|   null|
|              status|              string|   null|
|                    |                    |       |
|# Detailed Table ...|                    |       |
|            Database|           itv024128|       |
|               Table|           orders1gb|       |
|               Owner|           itv024128|       |
|        Created Time|Tue Jul 21 08:07:...|       |
|         Last Access|             UNKNOWN|       |
|          Created By|         Spark 3.1.2|       |
|                Type|             MANAGED|       |
|            Provider|                 csv|       |
|          Statistics|     840836625 bytes|       |
|            Location|hdfs://m01.itvers...|       |
|       Serd

In [24]:
spark.sql("insert into itv024128.orders1GB values(  11111, CAST('2013-07-25' AS DATE),  11599 , 'BOOKED')")
## a new file gets created in /user/itv024128/warehouse/itv024128.db/orders1gb

""


In [25]:
## previous cache gets invalidated as a new row was inserted

In [26]:
spark.sql("select distinct status from itv024128.orders1GB ").show() ## this will reload the cache

+---------------+
|         status|
+---------------+
|PENDING_PAYMENT|
|       COMPLETE|
|        ON_HOLD|
| PAYMENT_REVIEW|
|         BOOKED|
|     PROCESSING|
|         CLOSED|
|SUSPECTED_FRAUD|
|        PENDING|
|       CANCELED|
+---------------+



In [27]:
spark.sql("select distinct status from itv024128.orders1GB ").show()  ## will run fast with new cache

+---------------+
|         status|
+---------------+
|PENDING_PAYMENT|
|       COMPLETE|
|        ON_HOLD|
| PAYMENT_REVIEW|
|         BOOKED|
|     PROCESSING|
|         CLOSED|
|SUSPECTED_FRAUD|
|        PENDING|
|       CANCELED|
+---------------+



In [28]:
spark.sql("clear cache") ## removes entire cache, every tables

""
